# 09 — Cross-Validation of Tuned Candidates

## Objective

This notebook evaluates the two leading holdout configurations, T11 and T8, with the exact fold assignment used by the 04.5 reference model.

Reusing folds creates paired comparisons: every candidate is scored on the same validation customers in every fold. This is a stricter test than comparing averages from independently shuffled splits.


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """Use the parent of the notebook working directory as the project root."""
    project_root = Path.cwd().resolve().parent
    if not (project_root / 'data').is_dir():
        raise FileNotFoundError(
            "Expected the project data directory one level above the current working directory."
        )
    return project_root

import gc, json, platform, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
print('Python:',platform.python_version()); print('LightGBM:',lgb.__version__)


## 1. Define candidates and reuse the benchmark folds

T11 combines a 63-leaf tree, larger minimum leaf size, 80% column sampling, and stronger L1/L2 regularization. T8 uses the same tree structure with 70% column sampling and lighter regularization.

The saved reference OOF file is the source of truth for fold membership.


In [ ]:
ROOT=find_project_root()
STEP_DIR=ROOT/'09_tuned_cv'
DATA_PATH=ROOT/'04_5_temporal_refined'/'train_aggregated_temporal_refined.feather'
BASELINE_CV_METRICS_PATH=ROOT/'07_baseline_cv'/'lightgbm_04_5_cv_metrics.json'
BASELINE_OOF_PATH=ROOT/'07_baseline_cv'/'lightgbm_04_5_oof_predictions.parquet'
FOLD_RESULTS_PATH=STEP_DIR/'lightgbm_tuned_cv_fold_metrics.csv'
OOF_PATH=STEP_DIR/'lightgbm_tuned_cv_oof_predictions.parquet'
IMPORTANCE_PATH=STEP_DIR/'lightgbm_tuned_cv_feature_importance.csv'
METRICS_PATH=STEP_DIR/'lightgbm_tuned_cv_metrics.json'
for p in [DATA_PATH,BASELINE_CV_METRICS_PATH,BASELINE_OOF_PATH]: assert p.exists(),f'找不到：{p}'

CANDIDATES={
 'T11':dict(num_leaves=63,min_child_samples=200,subsample=1.0,subsample_freq=0,colsample_bytree=.8,reg_alpha=.1,reg_lambda=5.,max_bin=127),
 'T8':dict(num_leaves=63,min_child_samples=200,subsample=1.0,subsample_freq=0,colsample_bytree=.7,reg_alpha=0.,reg_lambda=1.,max_bin=127),
}
BASE_PARAMS=dict(objective='binary',n_estimators=2500,learning_rate=.03,max_depth=-1,random_state=42,n_jobs=4,verbosity=-1)
print(CANDIDATES)


## 2. Align customer rows exactly

The reference data and OOF assignment are checked for unique IDs, equal lengths, identical order, and complete folds. These assertions prevent a subtle row-order mismatch from assigning scores or folds to the wrong customer.


In [ ]:
data=pd.read_feather(DATA_PATH)
baseline_oof=pd.read_parquet(BASELINE_OOF_PATH)
assert data['customer_ID'].is_unique and baseline_oof['customer_ID'].is_unique
assert np.array_equal(data['customer_ID'].to_numpy(),baseline_oof['customer_ID'].to_numpy())
assert np.array_equal(data['target'].astype('int8').to_numpy(),baseline_oof['target'].astype('int8').to_numpy())
assert set(baseline_oof['fold'])=={1,2,3}
customer_ids=data['customer_ID'].copy(); y=data['target'].astype('int8').copy(); fold_assignment=baseline_oof['fold'].astype('int8').to_numpy()
X=data.drop(columns=['customer_ID','target']).copy(); del data; gc.collect()
X['S_2_last']=(pd.to_datetime(X['S_2_last'])-pd.Timestamp('2018-03-01')).dt.days.astype('int16')
CAT_VARS=['B_30','B_38','D_114','D_116','D_117','D_120','D_126','D_63','D_64','D_66','D_68']
CAT_FEATURES=[f'{c}_last' for c in CAT_VARS]
for c in CAT_FEATURES: X[c]=X[c].astype('category')
print(f'{len(X):,} customers, {X.shape[1]} features; exact 07 fold alignment passed')
display(pd.crosstab(fold_assignment,y,rownames=['fold'],colnames=['target']))


## 3. Define a checkpointed fold runner

Each candidate-fold pair verifies zero customer overlap, trains with early stopping, writes its fold model and predictions, and appends metrics to disk. Completed pairs can be skipped safely after an interruption.

The official metric implementation is identical to the reference CV notebook.


In [ ]:
def amex_metric(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=np.int8); y_pred=np.asarray(y_pred,dtype=np.float64); w=np.where(y_true==0,20.,1.)
    order=np.argsort(-y_pred,kind='mergesort'); yy,ww=y_true[order],w[order]; top=np.cumsum(ww)<=.04*ww.sum()
    top_four=(yy[top]*ww[top]).sum()/(y_true*w).sum()
    def gini(pred):
        idx=np.argsort(-pred,kind='mergesort'); sy,sw=y_true[idx],w[idx]
        return np.sum((np.cumsum(sy*sw)/np.sum(sy*sw)-np.cumsum(sw)/np.sum(sw))*sw)
    return .5*(gini(y_pred)/gini(y_true)+top_four)

RESULT_COLUMNS=['candidate','fold','train_customers','valid_customers','customer_overlap','positive_rate','roc_auc','amex_metric','best_iteration','training_seconds']
def load_results(): return pd.read_csv(FOLD_RESULTS_PATH) if FOLD_RESULTS_PATH.exists() else pd.DataFrame(columns=RESULT_COLUMNS)
if OOF_PATH.exists():
    oof=pd.read_parquet(OOF_PATH)
    assert np.array_equal(oof.customer_ID.to_numpy(),customer_ids.to_numpy())
else:
    oof=pd.DataFrame({'customer_ID':customer_ids.to_numpy(),'target':y.to_numpy(),'fold':fold_assignment,'T11_probability':np.nan,'T8_probability':np.nan})

def run_fold(candidate,fold):
    existing=load_results()
    if ((existing.candidate==candidate)&(existing.fold==fold)).any(): print('SKIP',candidate,fold); return
    train_idx=np.flatnonzero(fold_assignment!=fold); valid_idx=np.flatnonzero(fold_assignment==fold)
    train_ids=customer_ids.iloc[train_idx]; valid_ids=customer_ids.iloc[valid_idx]
    overlap=len(set(train_ids).intersection(set(valid_ids))); assert overlap==0
    params=BASE_PARAMS|CANDIDATES[candidate]; model=lgb.LGBMClassifier(**params)
    print(f'\nSTART {candidate} fold {fold}: train={len(train_idx):,}, valid={len(valid_idx):,}, overlap={overlap}')
    started=time.perf_counter()
    model.fit(X.iloc[train_idx],y.iloc[train_idx],eval_set=[(X.iloc[valid_idx],y.iloc[valid_idx])],eval_metric='auc',categorical_feature=CAT_FEATURES,callbacks=[lgb.early_stopping(100,verbose=True),lgb.log_evaluation(100)])
    probability=model.predict_proba(X.iloc[valid_idx],num_iteration=model.best_iteration_)[:,1]
    elapsed=time.perf_counter()-started
    row={'candidate':candidate,'fold':fold,'train_customers':len(train_idx),'valid_customers':len(valid_idx),'customer_overlap':overlap,'positive_rate':float(y.iloc[valid_idx].mean()),'roc_auc':float(roc_auc_score(y.iloc[valid_idx],probability)),'amex_metric':float(amex_metric(y.iloc[valid_idx],probability)),'best_iteration':int(model.best_iteration_),'training_seconds':float(elapsed)}
    pd.concat([existing,pd.DataFrame([row])],ignore_index=True).to_csv(FOLD_RESULTS_PATH,index=False)
    oof.loc[valid_idx,f'{candidate}_probability']=probability.astype('float32'); oof.to_parquet(OOF_PATH,index=False,compression='zstd')
    importance=pd.DataFrame({'candidate':candidate,'fold':fold,'feature':model.booster_.feature_name(),'gain':model.booster_.feature_importance(importance_type='gain'),'split':model.booster_.feature_importance(importance_type='split')})
    old_imp=pd.read_csv(IMPORTANCE_PATH) if IMPORTANCE_PATH.exists() else pd.DataFrame(columns=importance.columns)
    pd.concat([old_imp,importance],ignore_index=True).to_csv(IMPORTANCE_PATH,index=False)
    model.booster_.save_model(STEP_DIR/f'lightgbm_{candidate}_cv_fold_{fold}.txt',num_iteration=model.best_iteration_)
    print('DONE',row); del model,probability,importance; gc.collect()


## 4. Evaluate T11

T11 is trained across all three reference folds. Its stronger regularization is intended to support the larger 63-leaf trees without increasing out-of-sample variance.


In [ ]:
for fold in [1,2,3]: run_fold('T11',fold)


## 5. Evaluate T8

T8 is evaluated under the same protocol. Lower column sampling makes each tree cheaper and decorrelates trees more aggressively, providing a useful efficiency-versus-performance alternative.


In [ ]:
for fold in [1,2,3]: run_fold('T8',fold)


## 6. Compare folds and full OOF predictions

The final comparison includes fold mean and standard deviation as well as metrics computed from the concatenated OOF probabilities. The OOF value is used for model selection because it evaluates the full customer population out of sample.

T11 wins by a small but consistent margin over both T8 and the original 04.5 parameters. The gain is real enough to retain, but too small to justify a claim of a major modeling breakthrough.


In [ ]:
results=load_results().sort_values(['candidate','fold']).reset_index(drop=True)
assert len(results)==6 and (results.customer_overlap==0).all()
assert not oof[['T11_probability','T8_probability']].isna().any().any()
summaries=[]
for candidate in ['T11','T8']:
    part=results[results.candidate==candidate]
    pred=oof[f'{candidate}_probability'].to_numpy()
    summaries.append({'model':candidate,'fold_auc_mean':part.roc_auc.mean(),'fold_auc_std':part.roc_auc.std(ddof=1),'fold_amex_mean':part.amex_metric.mean(),'fold_amex_std':part.amex_metric.std(ddof=1),'oof_roc_auc':roc_auc_score(y,pred),'oof_amex_metric':amex_metric(y,pred),'best_iteration_mean':part.best_iteration.mean(),'best_iteration_std':part.best_iteration.std(ddof=1),'total_training_seconds':part.training_seconds.sum()})
baseline_json=json.loads(BASELINE_CV_METRICS_PATH.read_text()); b=baseline_json['summary']
baseline={'model':'04.5 original','fold_auc_mean':b['fold_roc_auc_mean'],'fold_auc_std':b['fold_roc_auc_std'],'fold_amex_mean':b['fold_amex_mean'],'fold_amex_std':b['fold_amex_std'],'oof_roc_auc':b['oof_roc_auc'],'oof_amex_metric':b['oof_amex_metric'],'best_iteration_mean':b['best_iteration_mean'],'best_iteration_std':b['best_iteration_std'],'total_training_seconds':b['total_training_seconds']}
comparison=pd.DataFrame([baseline,*summaries]).set_index('model')
comparison['delta_oof_auc_vs_original']=comparison.oof_roc_auc-baseline['oof_roc_auc']
comparison['delta_oof_amex_vs_original']=comparison.oof_amex_metric-baseline['oof_amex_metric']
display(results); display(comparison)

importance=pd.read_csv(IMPORTANCE_PATH)
importance_summary=importance.groupby(['candidate','feature'],as_index=False).agg(mean_gain=('gain','mean'),std_gain=('gain','std'),mean_split=('split','mean'))
importance_summary['gain_share']=importance_summary.groupby('candidate').mean_gain.transform(lambda x:x/x.sum())
importance_summary['rank']=importance_summary.groupby('candidate').mean_gain.rank(method='first',ascending=False).astype(int)
importance_summary=importance_summary.sort_values(['candidate','rank'])
display(importance_summary[importance_summary['rank']<=20])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Paired fold comparison and full-OOF result
fold_chart = pd.concat([
    pd.DataFrame(baseline_json['folds']).assign(candidate='04.5 original'),
    results.rename(columns={'candidate': 'candidate'}),
], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.pointplot(data=fold_chart, x='fold', y='roc_auc', hue='candidate', markers='o', ax=axes[0])
axes[0].set(title='ROC-AUC on identical validation folds', xlabel='Fold', ylabel='ROC-AUC')
sns.pointplot(data=fold_chart, x='fold', y='amex_metric', hue='candidate', markers='o', ax=axes[1])
axes[1].set(title='AMEX metric on identical validation folds', xlabel='Fold', ylabel='AMEX metric')
axes[0].legend(title='')
axes[1].legend(title='')
sns.despine()
plt.tight_layout()
plt.show()

oof_chart = comparison.reset_index().rename(columns={'index': 'model'})
fig, ax = plt.subplots(figsize=(8, 4.2))
oof_long = oof_chart.melt(id_vars='model', value_vars=['oof_roc_auc', 'oof_amex_metric'], var_name='metric', value_name='score')
sns.barplot(data=oof_long, x='metric', y='score', hue='model', ax=ax)
ax.set(title='T11 delivers the strongest full OOF result', xlabel='', ylabel='OOF score', ylim=(0.76, 0.97))
ax.legend(title='')
sns.despine()
plt.tight_layout()
plt.show()


## 7. Save the final experimental record

Fold metrics, full OOF predictions, averaged importance, and a structured comparison report are saved. T11 is selected as the final candidate for a future full-data fit; this notebook does not yet train on all customers or generate test predictions.


In [ ]:
winner=comparison.loc[['T11','T8']].sort_values(['oof_amex_metric','oof_roc_auc'],ascending=False).index[0]
output={'baseline_04_5':baseline,'candidates':summaries,'winner_by_oof_amex':winner,'comparison':comparison.reset_index().to_dict(orient='records'),'fold_assignment_source':str(BASELINE_OOF_PATH),'customer_level':True}
METRICS_PATH.write_text(json.dumps(output,indent=2,default=float),encoding='utf-8')
print('Winner by OOF AMEX:',winner)
for p in [FOLD_RESULTS_PATH,OOF_PATH,IMPORTANCE_PATH,METRICS_PATH,*[STEP_DIR/f'lightgbm_{c}_cv_fold_{f}.txt' for c in ['T11','T8'] for f in [1,2,3]]]: print(p.name,'->',p.resolve())
